<a href="https://colab.research.google.com/github/dewmindi/autoGrader/blob/dewmindi-do-2/AutoGrader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import csv
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from collections import Counter

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:

uploaded = files.upload()


data = []
with open('question4.csv', mode='r') as file:
    csvFile = csv.reader(file)
    header = next(csvFile)
    for lines in csvFile:
        data.append(lines)

Saving question4.csv to question4 (1).csv


In [ ]:
answer_index =0

def count_words(text):
    cleaned_text = text.replace(",", "").replace(".", "").strip()
    return len(word_tokenize(cleaned_text))

def count_unique_words(text):
    cleaned_text = text.replace(",", "").replace(".", "").strip()
    return len(set(word_tokenize(cleaned_text)))

def count_stop_words(text):
    cleaned_text = text.replace(",", "").replace(".", "").strip()
    stop_words_set = set(stopwords.words('english'))
    return sum(1 for word in word_tokenize(text) if word in stop_words_set)

def count_non_stop_words(text):
    cleaned_text = text.replace(",", "").replace(".", "").strip()
    return count_words(text) - count_stop_words(text)

def count_sentences(text):
    cleaned_text = text.replace(",", "").replace(".", "").strip()
    return len(sent_tokenize(text))

def count_long_words(text, length=5):
    cleaned_text = text.replace(",", "").replace(".", "").strip()
    return sum(1 for word in word_tokenize(text) if len(word) > length)

def unique_words_with_frequency(text):
    tokenized = word_tokenize(text)
    return dict(Counter(tokenized))


In [ ]:
processed_data = []
header = ['Answer', 'Words', 'Unique_Words', 'Stop_Words',
          'Non_Stop_Words', 'Sentences', 'Long_Words', 'Word_Frequencies']
processed_data.append(header)

for row in data:
    Answer = row[answer_index].lower().strip()
    Words = count_words(Answer)
    no_of_unique_words = count_unique_words(Answer)
    no_of_stop_words = count_stop_words(Answer)
    no_of_non_stop_words = count_non_stop_words(Answer)
    no_of_sentences = count_sentences(Answer)
    no_of_long_words = count_long_words(Answer)
    Word_Frequencies = unique_words_with_frequency(Answer)

    processed_data.append([Answer, Words, no_of_unique_words, no_of_stop_words,
                           no_of_non_stop_words, no_of_sentences, no_of_long_words, Word_Frequencies])

In [ ]:
import pandas as pd

for row in processed_data[1:]:
    row[-1] = str(row[-1])

output_file = "processed_dataset_question4.csv"
df = pd.DataFrame(processed_data[1:], columns=processed_data[0])
df.to_csv(output_file, index=False)

from google.colab import files
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
import pandas as pd

# Upload multiple CSV files
print("Please select multiple CSV files to upload:")
uploaded = files.upload()

# Initialize an empty list to store DataFrames
dataframes = []

# Loop through the uploaded files and read each into a DataFrame
for file_name in uploaded.keys():
    df = pd.read_csv(file_name)  # Read CSV file
    dataframes.append(df)        # Append DataFrame to the list

# Combine all DataFrames into one
combined_df = pd.concat(dataframes, ignore_index=True)

# Add the "Rating" column
combined_df['Rating'] = [(i % 10) + 1 for i in range(len(combined_df))]

# Save the combined DataFrame to a new CSV file
output_file = "AutoGrader_DataSet.csv"
combined_df.to_csv(output_file, index=False)

print(f"Combined CSV file with ratings saved as {output_file}")

# Provide a download link for the combined CSV
files.download(output_file)


Please select multiple CSV files to upload:


Saving processed_dataset_question1.csv to processed_dataset_question1 (2).csv
Saving processed_dataset_question2.csv to processed_dataset_question2 (2).csv
Saving processed_dataset_question3.csv to processed_dataset_question3 (2).csv
Saving processed_dataset_question4.csv to processed_dataset_question4 (2).csv
Saving processed_dataset_question5.csv to processed_dataset_question5 (2).csv
Combined CSV file with ratings saved as AutoGrader_DataSet.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
import pandas as pd
import csv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files  # For uploading and downloading files in Colab

# Step 1: Upload the CSV file
print("Please upload your dataset (CSV format):")
uploaded = files.upload()

# Get the file name of the uploaded file
file_name = list(uploaded.keys())[0]

# Read the uploaded CSV file
data = []
with open(file_name, mode='r') as file:
    csvFile = csv.reader(file)
    header = next(csvFile)
    for lines in csvFile:
        data.append(lines)

# Convert the data to a DataFrame
df = pd.DataFrame(data, columns=header)

# Step 2: Preprocess the text
def preprocess_text(text):
    """Preprocess text by converting to lowercase and stripping whitespace."""
    return str(text).lower().strip()

# Apply preprocessing to both columns
df['Real Answer'] = df['Real Answer'].apply(preprocess_text)
df['Answer'] = df['Answer'].apply(preprocess_text)

# Step 3: Compute TF-IDF vectors
# Initialize the TF-IDF vectorizer
vectorizer = TfidfVectorizer()

# Combine all text for a consistent vocabulary
all_text = pd.concat([df['Real Answer'], df['Answer']], axis=0)

# Fit and transform the text data
vectorizer.fit(all_text)
real_answer_tfidf = vectorizer.transform(df['Real Answer'])
answer_tfidf = vectorizer.transform(df['Answer'])

# Step 4: Compute Cosine Similarity
# Compute similarity for each pair of Real Answer and Answer
similarity_scores = [
    cosine_similarity(real_answer_tfidf[i], answer_tfidf[i])[0][0]
    for i in range(len(df))
]

# Step 5: Normalize the Rating and Adjust Similarity
# Assuming the rating is in the range [1, 10]
df['Rating Normalized'] = df['Rating'].astype(float) / 10.0

# Adjust similarity score by normalizing the rating
df['Adjusted Similarity'] = [
    similarity_scores[i] * df['Rating Normalized'][i]
    for i in range(len(df))
]

# Step 6: Save the updated dataset
output_file_name = "AutoGrader_DataSet_with_Adjusted_Similarity.csv"
df.to_csv(output_file_name, index=False)

print(f"Updated dataset with adjusted similarity scores saved to: {output_file_name}")

# Step 7: Download the updated dataset
files.download(output_file_name)


Please upload your dataset (CSV format):


Saving AutoGrader_DataSet_WithAnswers.csv to AutoGrader_DataSet_WithAnswers (1).csv
Updated dataset with adjusted similarity scores saved to: AutoGrader_DataSet_with_Adjusted_Similarity.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
import pandas as pd
import csv
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files  # For uploading and downloading files in Colab

# Step 1: Upload the CSV file
print("Please upload your dataset (CSV format):")
uploaded = files.upload()

# Get the file name of the uploaded file
file_name = list(uploaded.keys())[0]

# Read the uploaded CSV file
data = []
with open(file_name, mode='r') as file:
    csvFile = csv.reader(file)
    header = next(csvFile)
    for lines in csvFile:
        data.append(lines)

# Convert the data to a DataFrame
df = pd.DataFrame(data, columns=header)

# Step 2: Load the pre-trained Sentence-BERT model
model = SentenceTransformer('all-MiniLM-L6-v2')  # You can use other models if needed

# Step 3: Generate embeddings for Real Answer and Answer
df['Real Answer Embedding'] = df['Real Answer'].apply(lambda x: model.encode(str(x)))
df['Answer Embedding'] = df['Answer'].apply(lambda x: model.encode(str(x)))

# Step 4: Compute Cosine Similarity Between the Embeddings
similarity_scores = [
    cosine_similarity([df['Real Answer Embedding'][i]], [df['Answer Embedding'][i]])[0][0]
    for i in range(len(df))
]

# Step 5: Normalize the Rating (If rating exists in your dataset)
df['Rating Normalized'] = df['Rating'].astype(float) / 10.0

# Step 6: Adjust Similarity Based on Rating
df['Adjusted Similarity'] = [
    similarity_scores[i] * df['Rating Normalized'][i]
    for i in range(len(df))
]

# Step 7: Save the updated dataset with semantic similarity scores
output_file_name = "AutoGrader_DataSet_with_Semantic_Similarity.csv"
df.to_csv(output_file_name, index=False)

print(f"Updated dataset with adjusted semantic similarity scores saved to: {output_file_name}")

# Step 8: Download the updated dataset
files.download(output_file_name)


Please upload your dataset (CSV format):


Saving AutoGrader_DataSet_WithAnswers.csv to AutoGrader_DataSet_WithAnswers (2).csv


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Updated dataset with adjusted semantic similarity scores saved to: AutoGrader_DataSet_with_Semantic_Similarity.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>